## Model Evaluation

This section will compare the performances between 3 architectures of choice and evaluate them.

#### YOLO 11m

In [1]:
from ultralytics import YOLO

model = YOLO("../../runs/detect/yolo11m_COCO_V2/exp2/weights/best.pt")
metrics = model.val(
    data="../../final_unified_dataset/dataset.yaml",
    imgsz=640,
    split='test',
    plots=True,
)
print(metrics.speed)

Ultralytics 8.4.31  Python-3.13.3 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11m summary (fused): 126 layers, 20,031,574 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 183.3160.5 MB/s, size: 149.5 KB)
val: Scanning C:\Users\HP Victus\CDS_2026Spring_project\final_unified_dataset\labels\test.cache... 507 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 507/507 60.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 2.6it/s 12.1s0.4s
                   all        507       3034      0.847      0.743      0.827      0.634
                person        464       2336      0.891      0.822      0.902      0.704
                   bag        306        698      0.803      0.665      0.752      0.565
Speed: 1.7ms preprocess, 18.7ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\Users\HP Victus\runs\detect\val8
{'preprocess': 

#### Faster RCNN

In [2]:
import os
import time
import copy
from pathlib import Path

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
from PIL import Image

In [3]:
DATASET_ROOT = "../../final_unified_dataset"
SAVE_DIR = "./fasterrcnn_outputs"

In [4]:
# =========================
# CONFIG
# =========================


NUM_CLASSES = 3   # background + person + bag
NUM_EPOCHS = 1    # <- only 1 epoch
BATCH_SIZE = 2    # smaller batch is safer on Mac
LR = 0.0005       # lower LR to avoid NaNs
WEIGHT_DECAY = 0.0005
MOMENTUM = 0.9
NUM_WORKERS = 0   # Mac / notebook safe
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]

# Safer on Mac for Faster R-CNN
PREFER_MPS = False   # set True only if you really want to try MPS again

os.makedirs(SAVE_DIR, exist_ok=True)

In [5]:
# =========================
# DEVICE
# =========================
if PREFER_MPS and torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# =========================
# HELPERS
# =========================
def collate_fn(batch):
    return tuple(zip(*batch))

def get_image_files(folder):
    folder = Path(folder)
    files = []
    for ext in IMAGE_EXTENSIONS:
        files.extend(folder.glob(f"*{ext}"))
    return sorted(files)

def yolo_to_xyxy(x_center, y_center, w, h, img_w, img_h):
    x_center *= img_w
    y_center *= img_h
    w *= img_w
    h *= img_h

    x1 = x_center - w / 2
    y1 = y_center - h / 2
    x2 = x_center + w / 2
    y2 = y_center + h / 2

    x1 = max(0.0, min(x1, img_w - 1))
    y1 = max(0.0, min(y1, img_h - 1))
    x2 = max(0.0, min(x2, img_w - 1))
    y2 = max(0.0, min(y2, img_h - 1))

    return [x1, y1, x2, y2]

def parse_yolo_label_file(label_path, img_w, img_h):
    boxes = []
    labels = []
    areas = []
    iscrowd = []

    if not label_path.exists():
        return boxes, labels, areas, iscrowd

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            continue

        try:
            cls_id, x_c, y_c, w, h = map(float, parts)
        except ValueError:
            continue

        # basic YOLO sanity checks
        if not (0 <= cls_id <= 1):
            continue
        if not (0.0 <= x_c <= 1.0 and 0.0 <= y_c <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
            continue

        # Faster R-CNN labels: 0 is background, so shift by +1
        label = int(cls_id) + 1

        box = yolo_to_xyxy(x_c, y_c, w, h, img_w, img_h)
        x1, y1, x2, y2 = box

        bw = x2 - x1
        bh = y2 - y1

        # reject invalid or tiny boxes
        if bw <= 1 or bh <= 1:
            continue

        boxes.append(box)
        labels.append(label)
        areas.append(bw * bh)
        iscrowd.append(0)

    return boxes, labels, areas, iscrowd

Using device: cuda


In [6]:
# =========================
# DATASET
# =========================
class YOLODetectionDataset(Dataset):
    def __init__(self, root, split="train", transforms=None):
        self.root = Path(root)
        self.split = split
        self.transforms = transforms

        self.image_dir = self.root / "images" / split
        self.label_dir = self.root / "labels" / split

        all_image_files = get_image_files(self.image_dir)
        if len(all_image_files) == 0:
            raise ValueError(f"No images found in {self.image_dir}")

        # Keep only images with at least one valid box
        self.image_files = []
        dropped = 0

        for img_path in all_image_files:
            label_path = self.label_dir / f"{img_path.stem}.txt"
            try:
                with Image.open(img_path) as image:
                    img_w, img_h = image.size
                boxes, labels, areas, iscrowd = parse_yolo_label_file(label_path, img_w, img_h)
                if len(boxes) > 0:
                    self.image_files.append(img_path)
                else:
                    dropped += 1
            except Exception:
                dropped += 1

        if len(self.image_files) == 0:
            raise ValueError(f"No valid labeled images found in {self.image_dir}")

        print(f"[{split}] Kept {len(self.image_files)} valid images, dropped {dropped} invalid/empty images")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.label_dir / f"{img_path.stem}.txt"

        image = Image.open(img_path).convert("RGB")
        img_w, img_h = image.size

        boxes, labels, areas, iscrowd = parse_yolo_label_file(label_path, img_w, img_h)

        # Since invalid/empty files were filtered in __init__, this should be safe.
        # But keep a defensive fallback.
        if len(boxes) == 0:
            boxes = torch.tensor([[0.0, 0.0, 2.0, 2.0]], dtype=torch.float32)
            labels = torch.tensor([1], dtype=torch.int64)
            areas = torch.tensor([4.0], dtype=torch.float32)
            iscrowd = torch.tensor([0], dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        image_id = torch.tensor([idx], dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": areas,
            "iscrowd": iscrowd
        }

        image = F.to_tensor(image)

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target

In [7]:
# =========================
# MODEL
# =========================
def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
        in_features, num_classes
    )
    return model

# =========================
# TRAIN / EVAL
# =========================
def train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=20):
    model.train()
    running_loss = 0.0
    valid_steps = 0

    for i, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        try:
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            if not torch.isfinite(losses):
                loss_items = {k: float(v.item()) for k, v in loss_dict.items()}
                print(f"[WARNING] Non-finite loss at step {i}. Skipping batch. Losses: {loss_items}")
                optimizer.zero_grad(set_to_none=True)
                continue

            optimizer.zero_grad(set_to_none=True)
            losses.backward()

            # gradient clipping helps stop exploding updates
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()

            loss_value = float(losses.item())
            running_loss += loss_value
            valid_steps += 1

            if i % print_freq == 0:
                loss_items = {k: float(v.item()) for k, v in loss_dict.items()}
                print(
                    f"Epoch [{epoch}] Step [{i}/{len(data_loader)}] "
                    f"Total Loss: {loss_value:.4f} | {loss_items}"
                )

        except RuntimeError as e:
            print(f"[WARNING] Runtime error at step {i}. Skipping batch. Error: {e}")
            optimizer.zero_grad(set_to_none=True)
            continue

    if valid_steps == 0:
        return float("inf")
    return running_loss / valid_steps

@torch.no_grad()
def evaluate_loss(model, data_loader, device):
    model.train()  # needed for detection loss computation
    total_loss = 0.0
    valid_steps = 0

    for i, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        try:
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            if not torch.isfinite(losses):
                print(f"[WARNING] Non-finite val loss at step {i}. Skipping batch.")
                continue

            total_loss += float(losses.item())
            valid_steps += 1

        except RuntimeError as e:
            print(f"[WARNING] Runtime error during validation at step {i}. Skipping batch. Error: {e}")
            continue

    model.eval()

    if valid_steps == 0:
        return float("inf")
    return total_loss / valid_steps

In [8]:
import time
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision.ops import box_iou
from collections import defaultdict

# ----------------------------
# CONFIG
# ----------------------------
CKPT_PATH = "./fasterrcnn_outputs/best_fasterrcnn.pth"
IOU_THRESHOLDS = np.arange(0.50, 0.96, 0.05)
SCORE_THRESH_FOR_PR = 0.05   # keep low for eval
MAX_DETS = 100

# ----------------------------
# Recreate model and test loader
# Assumes these already exist in earlier cells:
# - NUM_CLASSES
# - device
# - collate_fn
# - YOLODetectionDataset
# - DATASET_ROOT
# - get_model()
# ----------------------------

test_dataset = YOLODetectionDataset(DATASET_ROOT, split="test")
test_loader = DataLoader(
    test_dataset,
    batch_size=1,   # best for clean timing/eval
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

model = get_model(NUM_CLASSES)
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

print(f"Loaded checkpoint: {CKPT_PATH}")
print(f"Evaluating on {len(test_dataset)} test images...")

# ----------------------------
# Helpers
# ----------------------------
def compute_ap(recall, precision):
    """
    COCO-style AP integration from precision-recall curve.
    """
    recall = np.asarray(recall)
    precision = np.asarray(precision)

    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])
    return ap

def match_predictions_to_gt(pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, iou_thr):
    """
    Greedy one-to-one matching per class for one image at a specific IoU threshold.
    Returns:
      pred_records: list of (score, is_tp, pred_label)
      gt_count_per_class: dict[class_id] = number of GT objects
    """
    pred_records = []
    gt_count_per_class = defaultdict(int)

    unique_classes = set(gt_labels.tolist()) | set(pred_labels.tolist())

    for cls in unique_classes:
        gt_mask = (gt_labels == cls)
        pr_mask = (pred_labels == cls)

        cls_gt_boxes = gt_boxes[gt_mask]
        cls_pred_boxes = pred_boxes[pr_mask]
        cls_pred_scores = pred_scores[pr_mask]

        gt_count_per_class[int(cls)] += len(cls_gt_boxes)

        if len(cls_pred_boxes) == 0:
            continue

        order = torch.argsort(cls_pred_scores, descending=True)
        cls_pred_boxes = cls_pred_boxes[order]
        cls_pred_scores = cls_pred_scores[order]

        matched_gt = torch.zeros(len(cls_gt_boxes), dtype=torch.bool)

        if len(cls_gt_boxes) > 0:
            ious = box_iou(cls_pred_boxes, cls_gt_boxes)
        else:
            ious = torch.zeros((len(cls_pred_boxes), 0), dtype=torch.float32)

        for i in range(len(cls_pred_boxes)):
            if ious.shape[1] == 0:
                pred_records.append((float(cls_pred_scores[i]), 0, int(cls)))
                continue

            best_iou, best_gt_idx = ious[i].max(dim=0)
            if best_iou >= iou_thr and not matched_gt[best_gt_idx]:
                matched_gt[best_gt_idx] = True
                pred_records.append((float(cls_pred_scores[i]), 1, int(cls)))
            else:
                pred_records.append((float(cls_pred_scores[i]), 0, int(cls)))

    return pred_records, gt_count_per_class

# ----------------------------
# Run inference once, cache outputs, and time it
# ----------------------------
all_outputs = []
all_targets = []
inference_times = []

with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(device) for img in images]

        start = time.perf_counter()
        outputs = model(images)
        if device.type == "cuda":
            torch.cuda.synchronize()
        end = time.perf_counter()

        inference_times.append((end - start) * 1000.0)  # ms

        out = outputs[0]
        tgt = targets[0]

        keep = out["scores"].cpu() >= SCORE_THRESH_FOR_PR

        pred_boxes = out["boxes"].detach().cpu()[keep][:MAX_DETS]
        pred_scores = out["scores"].detach().cpu()[keep][:MAX_DETS]
        pred_labels = out["labels"].detach().cpu()[keep][:MAX_DETS]

        gt_boxes = tgt["boxes"].detach().cpu()
        gt_labels = tgt["labels"].detach().cpu()

        all_outputs.append({
            "boxes": pred_boxes,
            "scores": pred_scores,
            "labels": pred_labels,
        })
        all_targets.append({
            "boxes": gt_boxes,
            "labels": gt_labels,
        })

# ----------------------------
# Compute AP / Precision / Recall across IoU thresholds
# ----------------------------
aps_per_iou = []
precisions_per_iou = []
recalls_per_iou = []

for iou_thr in IOU_THRESHOLDS:
    class_pred_records = defaultdict(list)   # class -> [(score, tp), ...]
    class_gt_counts = defaultdict(int)       # class -> gt total

    for out, tgt in zip(all_outputs, all_targets):
        pred_records, gt_count_per_class = match_predictions_to_gt(
            out["boxes"], out["scores"], out["labels"],
            tgt["boxes"], tgt["labels"],
            iou_thr=iou_thr
        )

        for score, is_tp, cls in pred_records:
            class_pred_records[cls].append((score, is_tp))

        for cls, n in gt_count_per_class.items():
            class_gt_counts[cls] += n

    class_aps = []
    class_precisions = []
    class_recalls = []

    eval_classes = sorted(set(class_gt_counts.keys()) | set(class_pred_records.keys()))
    eval_classes = [c for c in eval_classes if c != 0]  # exclude background if present

    for cls in eval_classes:
        preds = class_pred_records[cls]
        n_gt = class_gt_counts[cls]

        if n_gt == 0:
            continue

        if len(preds) == 0:
            class_aps.append(0.0)
            class_precisions.append(0.0)
            class_recalls.append(0.0)
            continue

        preds = sorted(preds, key=lambda x: x[0], reverse=True)
        tps = np.array([p[1] for p in preds], dtype=np.float32)
        fps = 1.0 - tps

        cum_tp = np.cumsum(tps)
        cum_fp = np.cumsum(fps)

        recall_curve = cum_tp / max(n_gt, 1)
        precision_curve = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)

        ap = compute_ap(recall_curve, precision_curve)
        final_precision = precision_curve[-1] if len(precision_curve) else 0.0
        final_recall = recall_curve[-1] if len(recall_curve) else 0.0

        class_aps.append(ap)
        class_precisions.append(final_precision)
        class_recalls.append(final_recall)

    aps_per_iou.append(np.mean(class_aps) if class_aps else 0.0)
    precisions_per_iou.append(np.mean(class_precisions) if class_precisions else 0.0)
    recalls_per_iou.append(np.mean(class_recalls) if class_recalls else 0.0)

# ----------------------------
# Final metrics
# ----------------------------
map50 = aps_per_iou[0]                        # IoU = 0.50
map50_95 = float(np.mean(aps_per_iou))        # mean over 0.50:0.95
mean_precision = float(np.mean(precisions_per_iou))
mean_recall = float(np.mean(recalls_per_iou))
inference_ms = float(np.mean(inference_times))

print("\n===== Evaluation Results =====")
print(f"mAP50      : {map50:.4f}")
print(f"mAP50-95   : {map50_95:.4f}")
print(f"mAP        : {map50_95:.4f}")   # same as COCO-style mAP
print(f"Precision  : {mean_precision:.4f}")
print(f"Recall     : {mean_recall:.4f}")
print(f"inference_ms/image: {inference_ms:.2f}")

print("\nAP by IoU threshold:")
for thr, ap in zip(IOU_THRESHOLDS, aps_per_iou):
    print(f"IoU {thr:.2f}: AP={ap:.4f}")

[test] Kept 507 valid images, dropped 0 invalid/empty images
Loaded checkpoint: ./fasterrcnn_outputs/best_fasterrcnn.pth
Evaluating on 507 test images...

===== Evaluation Results =====
mAP50      : 0.6188
mAP50-95   : 0.3477
mAP        : 0.3477
Precision  : 0.0839
Recall     : 0.5124
inference_ms/image: 62.77

AP by IoU threshold:
IoU 0.50: AP=0.6188
IoU 0.55: AP=0.5880
IoU 0.60: AP=0.5520
IoU 0.65: AP=0.4979
IoU 0.70: AP=0.4324
IoU 0.75: AP=0.3400
IoU 0.80: AP=0.2549
IoU 0.85: AP=0.1446
IoU 0.90: AP=0.0455
IoU 0.95: AP=0.0026


#### SSD

In [9]:
!pip install ultralytics
!pip install tqdm
!pip install torchmetrics
!pip install pycocotools
!pip install faster-coco-eval

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import time
from pathlib import Path
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from torchvision.models.detection import ssd300_vgg16
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

C:\Users\HP Victus\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\HP Victus\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# SSD model setup (local)
class SSDDataset(Dataset):
    def __init__(self, root, split="train"):
        self.root = Path(root)
        self.img_dir = self.root / f"images/{split}"
        self.label_dir = self.root / f"labels/{split}"

        self.images = list(self.img_dir.glob("*.jpg"))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label_path = self.label_dir / (img_path.stem + ".txt")

        image = cv2.imread(str(img_path))
        h, w = image.shape[:2]
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    cls, x, y, bw, bh = map(float, line.strip().split())

                    x_min = (x - bw/2) * w
                    y_min = (y - bh/2) * h
                    x_max = (x + bw/2) * w
                    y_max = (y + bh/2) * h

                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(cls) + 1)
                    # +1 because 0 = background in SSD

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        image = torch.tensor(image / 255.0, dtype=torch.float32).permute(2, 0, 1)

        return image, target

In [3]:
from pathlib import Path
import torch
import time
from torchvision.models.detection import ssd300_vgg16
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

root = Path("../../final_unified_dataset")

val_dataset = SSDDataset(root, "test")
val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=lambda x: tuple(zip(*x))
)

model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3

model.load_state_dict(torch.load(
    "../../runs/detect/SSDv3_COCO_final/train2/weights/best.pth",
    map_location=device
))

model.to(device)
model.eval()

map_metric = MeanAveragePrecision(
    iou_thresholds=[0.5],
    class_metrics=True,
    backend="faster_coco_eval"
)

inference_times = []
total_tp = 0
total_fp = 0

# --- IoU helper ---
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Evaluating SSD"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # --- Timing ---
        if device.type == "cuda":
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            outputs = model(images)
            end.record()

            torch.cuda.synchronize()
            inference_times.append(start.elapsed_time(end))
        else:
            t0 = time.time()
            outputs = model(images)
            t1 = time.time()
            inference_times.append((t1 - t0) * 1000)

        map_metric.update(outputs, targets)

        # --- Precision calculation ---
        for out, tgt in zip(outputs, targets):
            pred_boxes = out["boxes"].cpu()
            gt_boxes = tgt["boxes"].cpu()

            for pb in pred_boxes:
                best_iou = 0

                for gb in gt_boxes:
                    iou = compute_iou(pb, gb)
                    best_iou = max(best_iou, iou)

                if best_iou >= 0.5:
                    total_tp += 1
                else:
                    total_fp += 1

# --- Metrics ---
map_results = map_metric.compute()

precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0

ssd_results = {
    "mAP50": map_results["map_50"].item(),
    "mAP50-95": map_results["map"].item(),
    "mean_recall": map_results["mar_100"].item(),
    "precision": precision,
    "inference_ms": sum(inference_times) / len(inference_times),
}

print("\nSSD evaluation results:")
for k, v in ssd_results.items():
    print(f"  {k}: {v:.4f}")

Evaluating SSD:   0%|          | 0/254 [00:00<?, ?it/s]C:\Users\HP Victus\AppData\Roaming\Python\Python313\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
Evaluating SSD: 100%|██████████| 254/254 [01:10<00:00,  3.62it/s]



SSD evaluation results:
  mAP50: 0.3193
  mAP50-95: 0.3193
  mean_recall: 0.6089
  precision: 0.0268
  inference_ms: 84.8230
